# 🎙️ Proyecto Auditoria DIGI — Fase 2: Transcripción

**Nodos cubiertos:** 5. Whisper Medium → 8. WhisperX (alineación) → 9. pyannote (diarización)

**Entrada:** `audio_vad.wav` generado por Fase 1  
**Salida:** guión diarizado con timestamps por interlocutor

> ℹ️ Nodos 6 (Qwen3-ASR) y 7 (Reconciliador) se añaden en versión final Docker — en v1 usamos solo Whisper.

---
**⚠️ IMPORTANTE:** Necesitas tu token de HuggingFace para pyannote. Se introduce en el Paso 2 — **nunca lo guardes en el código**.

## ✅ Paso 0 — Verificar GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ GPU disponible:')
    for linea in r.stdout.split('\n'):
        if any(x in linea for x in ['Tesla','RTX','A100','T4','V100','L4']):
            print(' ', linea.strip()); break
else:
    print('❌ Sin GPU — ve a: Entorno de ejecución → Cambiar tipo → GPU')

## 📦 Paso 1 — Instalar dependencias
Tarda ~3 minutos. No requiere reinicio.

In [ ]:
# WhisperX — incluye Whisper + alineación wav2vec2
!pip install -q git+https://github.com/m-bain/whisperX.git

# pyannote — diarización (quién habla cuándo)
!pip install -q pyannote.audio

# Audio utils
!pip install -q torchaudio soundfile

print('\n✅ Todo instalado')

## 🔑 Paso 2 — Token de HuggingFace
Introduce tu token aquí. Solo se guarda en memoria durante esta sesión.

In [ ]:
import getpass

HF_TOKEN = getpass.getpass('🔑 Introduce tu token de HuggingFace (hf_...): ')

if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 20:
    print('✅ Token recibido correctamente')
else:
    print('⚠️ El token no parece válido — comprueba que empieza por hf_')

## 🎵 Paso 3 — Cargar audio de Fase 1
Si vienes directamente de Fase 1 en la misma sesión, el archivo ya está disponible. Si es una sesión nueva, súbelo aquí.

In [ ]:
import os

ARCHIVO_AUDIO = 'audio_vad.wav'

if os.path.exists(ARCHIVO_AUDIO):
    duracion = os.path.getsize(ARCHIVO_AUDIO) / 1024
    print(f'✅ Audio de Fase 1 encontrado: {ARCHIVO_AUDIO} ({duracion:.0f} KB)')
else:
    print('📂 No se encontró audio_vad.wav — sube el archivo de Fase 1:')
    from google.colab import files
    subidos = files.upload()
    ARCHIVO_AUDIO = list(subidos.keys())[0]
    print(f'✅ Archivo recibido: {ARCHIVO_AUDIO}')

## 🎤 Nodo 5 — Whisper Medium: Transcripción

**¿Qué hace?** Convierte el audio en texto. Usamos el modelo Medium que da buen equilibrio entre velocidad y precisión. En Docker final usaremos Large V3.

In [ ]:
import whisperx
import gc, torch

DEVICE = 'cuda'
COMPUTE_TYPE = 'int8'  # Ahorra VRAM, suficiente para T4
IDIOMA = 'es'

print('🎤 Cargando Whisper Medium...')
modelo_asr = whisperx.load_model('medium', DEVICE, compute_type=COMPUTE_TYPE, language=IDIOMA)

print('   Transcribiendo audio... (puede tardar 2-4 min para una llamada de 20 min)')
audio = whisperx.load_audio(ARCHIVO_AUDIO)
resultado = modelo_asr.transcribe(audio, batch_size=8, language=IDIOMA)

# Liberar VRAM antes del siguiente modelo
del modelo_asr
gc.collect()
torch.cuda.empty_cache()

n_segmentos = len(resultado['segments'])
texto_total = ' '.join([s['text'] for s in resultado['segments']])
n_palabras = len(texto_total.split())

print(f'\n✅ Nodo 5 completado')
print(f'   Segmentos detectados: {n_segmentos}')
print(f'   Palabras transcritas: {n_palabras}')
print(f'\n   Vista previa (primeros 300 caracteres):')
print(f'   {texto_total[:300]}...')

## ⏱️ Nodo 8 — WhisperX: Alineación por palabra

**¿Qué hace?** Asigna un timestamp exacto a cada palabra — necesario para saber con precisión cuándo habla cada persona.

In [ ]:
print('⏱️ Cargando modelo de alineación wav2vec2...')
modelo_align, metadata = whisperx.load_align_model(language_code=IDIOMA, device=DEVICE)

print('   Alineando palabras...')
resultado_alineado = whisperx.align(
    resultado['segments'], modelo_align, metadata, audio, DEVICE,
    return_char_alignments=False
)

del modelo_align
gc.collect()
torch.cuda.empty_cache()

# Contar palabras con timestamp
palabras_con_ts = sum(
    len(s.get('words', [])) for s in resultado_alineado['segments']
)

print(f'\n✅ Nodo 8 completado')
print(f'   Palabras con timestamp: {palabras_con_ts}')

# Vista previa de un segmento alineado
if resultado_alineado['segments']:
    seg = resultado_alineado['segments'][0]
    print(f'   Ejemplo segmento 1: [{seg["start"]:.1f}s → {seg["end"]:.1f}s] "{seg["text"].strip()}"')

## 👥 Nodo 9 — pyannote: Diarización (quién habla cuándo)

**¿Qué hace?** Separa automáticamente las voces del agente y del cliente, asignando cada fragmento a un interlocutor (SPEAKER_00, SPEAKER_01...).

In [ ]:
print('👥 Cargando pyannote speaker-diarization-3.1...')
modelo_diar = whisperx.DiarizationPipeline(
    use_auth_token=HF_TOKEN,
    device=DEVICE
)

print('   Separando interlocutores... (1-2 min)')
segmentos_diar = modelo_diar(audio)

# Asignar interlocutor a cada segmento de texto
resultado_final = whisperx.assign_word_speakers(segmentos_diar, resultado_alineado)

# Contar intervenciones por interlocutor
speakers = {}
for seg in resultado_final['segments']:
    sp = seg.get('speaker', 'DESCONOCIDO')
    speakers[sp] = speakers.get(sp, 0) + 1

print(f'\n✅ Nodo 9 completado')
print(f'   Interlocutores detectados: {len(speakers)}')
for sp, n in sorted(speakers.items()):
    print(f'   {sp}: {n} intervenciones')

## 📝 Formatear guión final

Genera el guión diarizado listo para pasar al análisis de IA en Fase 3.

In [ ]:
import json

# Construir guión formateado
guion_lineas = []
speaker_anterior = None
bloque_actual = ''

for seg in resultado_final['segments']:
    speaker = seg.get('speaker', 'DESCONOCIDO')
    texto = seg['text'].strip()
    inicio = seg.get('start', 0)

    if speaker != speaker_anterior:
        if bloque_actual:
            guion_lineas.append({'speaker': speaker_anterior, 'texto': bloque_actual.strip()})
        bloque_actual = texto + ' '
        speaker_anterior = speaker
    else:
        bloque_actual += texto + ' '

if bloque_actual:
    guion_lineas.append({'speaker': speaker_anterior, 'texto': bloque_actual.strip()})

# Mostrar vista previa
print('📝 Vista previa del guión (primeras 15 intervenciones):')
print('=' * 60)
for item in guion_lineas[:15]:
    print(f"[{item['speaker']}]: {item['texto'][:120]}")
print('=' * 60)
print(f'\nTotal intervenciones: {len(guion_lineas)}')

# Texto plano para el LLM (Fase 3)
GUION_TEXTO = '\n'.join([f"[{l['speaker']}]: {l['texto']}" for l in guion_lineas])

# Guardar archivos
with open('guion_diarizado.txt', 'w', encoding='utf-8') as f:
    f.write(GUION_TEXTO)

resultado_fase2 = {
    'archivo_audio': ARCHIVO_AUDIO,
    'n_segmentos_whisper': n_segmentos,
    'n_palabras': n_palabras,
    'n_interlocutores': len(speakers),
    'intervenciones_por_speaker': speakers,
    'n_intervenciones_total': len(guion_lineas),
    'archivo_guion': 'guion_diarizado.txt'
}

with open('resultado_fase2.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_fase2, f, indent=2, ensure_ascii=False)

print('\n✅ Archivos guardados:')
print('   guion_diarizado.txt  — texto para Fase 3 (análisis IA)')
print('   resultado_fase2.json — metadatos')
print(f'\n➡️  Listo para Fase 3: análisis con Qwen3-8B')